# Partitioning, Bucketing & Writing Data

Designing how Spark saves data for efficiency and scalability

By this point, you’ve read data into Spark, transformed it, grouped it, even cleaned it. But data pipelines aren’t just about processing in memory — they’re about storing results back in a way that others can consume.

Think about it: if you process billions of rows every night and dump them all into a single file, the next analyst who queries it will have to scan everything, even if they only need last week’s data. That’s wasteful and slow.

The way we write data determines not just storage, but also query performance for everyone downstream. And Spark gives us two powerful tools to control this: partitioning and bucketing.

### Writing Data: The Basics
Let’s first remind ourselves how writing works in Spark.

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("day12-dates").getOrCreate()

df = spark.read.option("header", True).option("inferSchema", True).csv("/Volumes/thedataengineering_00/data/sampledata/12_sales_csv.csv")
df.write.mode("overwrite").parquet("/Volumes/thedataengineering_00/data/sampledata/sales_parquet")

In [0]:
df_output =  spark.read.parquet("/Volumes/thedataengineering_00/data/sampledata/sales_parquet")
display(df_output)

This saves your DataFrame as Parquet files in the specified directory. By default, Spark will create multiple files, one per partition in the DataFrame. If your DataFrame has 200 partitions, you’ll see 200 files. If it has 1 partition, you’ll see just 1 file.

This default behavior already hints at why we need to be deliberate about partitioning.

### Partitioning: Organizing Data by Columns
Partitioning is like creating labeled folders for your data. Instead of dumping everything into one pile, you tell Spark to organize output by one or more columns.

Imagine you have sales data across multiple regions and dates. If analysts often query “give me sales for North region in September”, wouldn’t it be nice if Spark could skip irrelevant files and read only what’s needed? That’s what partitioning does.

In [0]:
df.write \
  .partitionBy("region", "date") \
  .mode("overwrite") \
  .parquet("/Volumes/thedataengineering_00/data/sampledata/sales_partitioned")

Now, when someone queries only region=North, Spark’s engine (or any query engine like Hive, Presto, Databricks SQL) can skip scanning South, East, West folders altogether. This is called partition pruning.

Partitioning shines when:

- The column has low cardinality (few unique values, like region or month).
- Queries frequently filter on that column.

Partitioning performs poorly when:

The column has too many unique values (like customer_id with millions of distinct entries). You’ll end up with millions of tiny folders, which slows down metadata handling.

### Bucketing: Organizing Data Within a Column
Partitioning is folder-level organization. Bucketing is file-level organization.

Suppose you often join your sales table with a customer table on cust_id. Even if you partition by region, the join still requires Spark to shuffle data across nodes to line up matching customer IDs.

Bucketing lets you tell Spark: “Hey, let’s hash values of cust_id into, say, 10 buckets. Store rows with the same hash in the same file.” That way, when two bucketed tables join on the same column with the same number of buckets, Spark knows matching keys are colocated — no big shuffle required.

In [0]:
df.write \
  .bucketBy(10, "cust_id") \
  .sortBy("cust_id") \
  .mode("overwrite") \
  .saveAsTable("sales_bucketed")

This creates a Hive-style managed table where Spark guarantees the bucketing.

bucketBy(10, "cust_id") → splits data into 10 files by hashing customer IDs.

sortBy("cust_id") → sorts within each bucket, making joins and range queries faster.

Bucketing shines when:

- You have large joins on the same key repeatedly.
- You want to reduce shuffle cost.

But bucketing requires managed tables and consistency across datasets (same number of buckets, same hash function). So it’s more planning-intensive than partitioning.

Partitioning vs Bucketing
Think of partitioning as filing cabinets and bucketing as folders inside each drawer.

- Partitioning: skip entire sets of data quickly. Great for filters like WHERE region = 'North'.
- Bucketing: optimize joins and aggregations on specific keys. Great for repeated queries like “join sales with customers on cust_id.”

In many production setups, you combine them: partition by date, bucket by customer_id. That way, analysts can filter by date efficiently, and joins don’t overwhelm the cluster.

### File Sizes and the “Small Files Problem”
One practical detail: Spark writes one file per partition. If you have 2000 partitions but each only has a few rows, you’ll create thousands of tiny files. Query engines choke on this because opening files has overhead.

On the other hand, if you have 1 huge file, parallelism suffers — only one task can process it at a time.

That’s why tuning the number of output files matters. You can control this with repartition() or coalesce() before writing:

In [0]:
# Reduce to 50 partitions before writing (reasonable file size)
df.repartition(50).write.mode("overwrite").parquet("/Volumes/thedataengineering_00/data/sampledata/sales_balanced")

This way, you balance between parallelism and file size.

### Why This Matters for You as a Data Engineer
Partitioning, bucketing, and write strategies aren’t academic — they directly impact speed, cost, and usability:

- A poorly partitioned dataset can mean hours of query time wasted scanning irrelevant files.
- A well-bucketed join can save thousands in compute by eliminating shuffles.
- Balanced output files prevent downstream systems from being bottlenecked.

In short, you’re not just “writing data” — you’re designing data layouts that decide how efficient the entire pipeline ecosystem will be.

Wrapping Up
Today, we went deeper into how Spark saves data:

- Partitioning organizes files by column values, enabling pruning and fast filters.
- Bucketing distributes data evenly across files for efficient joins.
- File sizing (repartitioning/coalescing) avoids the small-files problem and improves parallelism.

Together, these techniques turn raw transformations into production-ready datasets that analysts, ML models, and BI dashboards can consume quickly and reliably.

 That’s Day 15. You now understand the art of writing data with partitioning and bucketing.

Next up, Day 16: Spark Execution Model — DAGs, Stages, and Tasks — where we’ll open Spark’s black box and learn what actually happens when your job runs.